In [1]:
# ---- CELDA 1: Evaluación de clasificación — setup + mapeo equipo->lado ----
import json, csv, configparser
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter

REPO_ROOT = Path("../..").resolve()
CLIP = "SNMOT-119"

JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_soccernet_{CLIP}.json"
with open(JSON_IN, encoding="utf-8") as fh:
    tu = json.load(fh)
tus_frames = tu["frames"]

GT_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gt" / "gt.txt"
gt_por_frame = defaultdict(list)
with open(GT_FILE) as fh:
    for fila in csv.reader(fh):
        fr, tid = int(fila[0]), int(fila[1])
        x,y,w,h = float(fila[2]),float(fila[3]),float(fila[4]),float(fila[5])
        gt_por_frame[fr].append((tid, x, y, w, h))

# Etiqueta fina de cada track del GT
INFO_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gameinfo.ini"
cfg = configparser.ConfigParser(); cfg.read(INFO_FILE)
def etiqueta_gt(desc):
    d = desc.lower()
    if "ball" in d: return "balon"
    if "referee" in d: return "arbitro"
    if "goalkeeper" in d: return "portero_izq" if "left" in d else "portero_der"
    if "player" in d: return "jugador_izq" if "left" in d else "jugador_der"
    return "otro"
etq_gt = {}
for key in cfg["Sequence"]:
    if key.startswith("trackletid_"):
        etq_gt[int(key.split("_")[1])] = etiqueta_gt(cfg["Sequence"][key])

# MAPEO equipo 0/1 -> izq/der (confirmado por el portero: equipo 0 = izq)
MAPEO_LADO = {0: "izq", 1: "der"}
print(f"Mapeo equipo->lado: {MAPEO_LADO}")

# Etiqueta fina de cada detección TUYA
def etiqueta_tuya(d):
    rol = d.get("rol"); eq = d.get("equipo")
    if rol == "balon": return "balon"
    if rol == "arbitro": return "arbitro"
    if rol == "portero":
        # el portero hereda el lado de su equipo
        return f"portero_{MAPEO_LADO.get(eq, '?')}"
    if rol == "jugador" and eq in (0,1):
        return f"jugador_{MAPEO_LADO[eq]}"
    return "otro"

print("Setup listo.")

Mapeo equipo->lado: {0: 'izq', 1: 'der'}
Setup listo.


In [2]:
# ---- CELDA 2: Emparejar detecciones (IoU) y construir matriz de confusión ----
def iou_xywh(a, b):
    ax1,ay1,ax2,ay2 = a[0],a[1],a[0]+a[2],a[1]+a[3]
    bx1,by1,bx2,by2 = b[0],b[1],b[0]+b[2],b[1]+b[3]
    ix1,iy1,ix2,iy2 = max(ax1,bx1),max(ay1,by1),min(ax2,bx2),min(ay2,by2)
    if ix2<ix1 or iy2<iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1)
    return inter/(a[2]*a[3]+b[2]*b[3]-inter)

CATEGORIAS = ["jugador_izq","jugador_der","portero_izq","portero_der","arbitro","balon"]
# matriz[gt][pred]
matriz = defaultdict(lambda: defaultdict(int))
emparejados = 0

for i, tus_dets in enumerate(tus_frames):
    gt_frame = i + 1
    gt_list = [(tid,(x,y,w,h)) for (tid,x,y,w,h) in gt_por_frame.get(gt_frame, [])]
    gt_usado = [False]*len(gt_list)

    for d in tus_dets:
        x1,y1,x2,y2 = d["bbox"]
        tb = (x1,y1,x2-x1,y2-y1)
        et_tu = etiqueta_tuya(d)
        # emparejar con el GT de mayor IoU
        mejor, mejor_j = 0.5, -1
        for j,(tid,gb) in enumerate(gt_list):
            if gt_usado[j]: continue
            v = iou_xywh(tb, gb)
            if v > mejor: mejor, mejor_j = v, j
        if mejor_j >= 0:
            tid = gt_list[mejor_j][0]
            et_gt = etq_gt.get(tid, "otro")
            matriz[et_gt][et_tu] += 1
            gt_usado[mejor_j] = True
            emparejados += 1

print(f"Detecciones emparejadas (IoU>=0.5): {emparejados}\n")

# Imprimir matriz
print("=== MATRIZ DE CONFUSIÓN (filas=GT, columnas=tu predicción) ===\n")
cols = CATEGORIAS
print(f"{'GT \\ TU':>14} | " + " ".join(f"{c[:8]:>9}" for c in cols))
print("-"*(14+3+10*len(cols)))
for gt_cat in CATEGORIAS:
    fila = matriz.get(gt_cat, {})
    total = sum(fila.values())
    if total == 0: continue
    print(f"{gt_cat:>14} | " + " ".join(f"{fila.get(c,0):>9}" for c in cols))

Detecciones emparejadas (IoU>=0.5): 5289

=== MATRIZ DE CONFUSIÓN (filas=GT, columnas=tu predicción) ===

       GT \ TU |  jugador_  jugador_  portero_  portero_   arbitro     balon
-----------------------------------------------------------------------------
   jugador_izq |       142      1980         0         0         0         0
   jugador_der |      2609       169         0         0         0         0
   portero_izq |       255         1         0         0         0         0
       arbitro |       126         0         0         0         0         0
         balon |         0         0         0         0         0         7


In [3]:
# ---- CELDA 3: Accuracy global y por categoría ----
total = sum(sum(f.values()) for f in matriz.values())
aciertos = sum(matriz[c].get(c,0) for c in CATEGORIAS)
print(f"=== EXACTITUD DE CLASIFICACIÓN ===\n")
print(f"Accuracy global: {aciertos}/{total} = {100*aciertos/total:.1f}%")
print(f"(de las detecciones emparejadas, cuántas clasificas con la etiqueta correcta)\n")

print("Por categoría (precision = de lo que predices así, cuánto acierta):")
for c in CATEGORIAS:
    # recall: de los GT de clase c, cuántos clasificaste como c
    gt_total = sum(matriz[c].values())
    tp = matriz[c].get(c, 0)
    # precision: de los que predijiste c, cuántos eran c
    pred_total = sum(matriz[g].get(c,0) for g in CATEGORIAS)
    rec = tp/gt_total if gt_total>0 else 0
    prec = tp/pred_total if pred_total>0 else 0
    if gt_total>0 or pred_total>0:
        print(f"  {c:>14}: precision={prec:.2f} recall={rec:.2f}  (GT={gt_total}, pred={pred_total}, ok={tp})")

# Matriz simplificada equipos: ¿confunde izq con der?
print("\n=== ¿Confunde EQUIPOS (izq vs der)? ===")
jug_ii = matriz["jugador_izq"].get("jugador_izq",0)
jug_id = matriz["jugador_izq"].get("jugador_der",0)
jug_dd = matriz["jugador_der"].get("jugador_der",0)
jug_di = matriz["jugador_der"].get("jugador_izq",0)
print(f"  Jugador izq clasificado izq: {jug_ii}, como der: {jug_id}")
print(f"  Jugador der clasificado der: {jug_dd}, como izq: {jug_di}")
tot_jug = jug_ii+jug_id+jug_dd+jug_di
if tot_jug>0:
    print(f"  Acierto de equipo en jugadores: {100*(jug_ii+jug_dd)/tot_jug:.1f}%")

=== EXACTITUD DE CLASIFICACIÓN ===

Accuracy global: 318/5289 = 6.0%
(de las detecciones emparejadas, cuántas clasificas con la etiqueta correcta)

Por categoría (precision = de lo que predices así, cuánto acierta):
     jugador_izq: precision=0.05 recall=0.07  (GT=2122, pred=3132, ok=142)
     jugador_der: precision=0.08 recall=0.06  (GT=2778, pred=2150, ok=169)
     portero_izq: precision=0.00 recall=0.00  (GT=256, pred=0, ok=0)
         arbitro: precision=0.00 recall=0.00  (GT=126, pred=0, ok=0)
           balon: precision=1.00 recall=1.00  (GT=7, pred=7, ok=7)

=== ¿Confunde EQUIPOS (izq vs der)? ===
  Jugador izq clasificado izq: 142, como der: 1980
  Jugador der clasificado der: 169, como izq: 2609
  Acierto de equipo en jugadores: 6.3%
